## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [1]:
#include <bits/stdc++.h>
using namespace std;

using ll = long long;

const int LIMIT = 1024 + 5;

int n, a, b;
int blockSize;
int permArr[LIMIT], where[LIMIT];

vector<int> resultOps;

int fastRead() {
    int num = 0;
    char c = getchar();

    while (c < '0' || c > '9') {
        c = getchar();
    }

    while (c >= '0' && c <= '9') {
        num = num * 10 + c - '0';
        c = getchar();
    }

    return num;
}

void refreshPosition() {
    for (int i = 0; i < n; ++i) {
        where[permArr[i]] = i;
    }
}

void doExchangeAB() {
    resultOps.push_back(0);

    for (int i = 0; i < n; ++i) {
        if (permArr[i] == a) {
            permArr[i] = b;
        } else if (permArr[i] == b) {
            permArr[i] = a;
        }
    }

    refreshPosition();
}

void doAdd(int shift) {
    shift %= n;
    if (shift < 0) {
        shift += n;
    }

    if (shift == 0) {
        return;
    }

    resultOps.push_back(shift);

    for (int i = 0; i < n; ++i) {
        permArr[i] = (permArr[i] + shift) % n;
    }

    refreshPosition();
}

void doXor(int mask) {
    if (mask == 0) {
        return;
    }

    resultOps.push_back(-mask);

    for (int i = 0; i < n; ++i) {
        permArr[i] ^= mask;
    }

    refreshPosition();
}

void calcPairPosition(int x, int y, int &posX, int &posY) {
    int diff = (y - x + n - blockSize + n) % n;

    posX = 0;
    posY = 0;

    for (int len = n / 2; len >= 2 * blockSize; len >>= 1) {
        if (diff >= len) {
            diff -= len;
            posY += len / 2;
        } else {
            posX += len / 2;
        }
    }

    int offset = x & (blockSize - 1);

    posX += n / 2 + offset;
    posY += offset;
}

void swapByOperations(int x, int y) {
    int gx = x / blockSize;
    int gy = y / blockSize;

    if ((gx & 1) == (gy & 1)) {
        int helper;

        if ((gx & 1) == 0) {
            helper = (x & (blockSize - 1)) + blockSize;
        } else {
            helper = x & (blockSize - 1);
        }

        swapByOperations(x, helper);
        swapByOperations(y, helper);
        swapByOperations(x, helper);
        return;
    }

    int pa, pb;
    int px, py;

    calcPairPosition(a, b, pa, pb);
    calcPairPosition(x, y, px, py);

    doAdd((px - x + n) % n);
    doXor(px ^ pa);
    doAdd((a - pa + n) % n);

    doExchangeAB();

    doAdd((pa - a + n) % n);
    doXor(px ^ pa);
    doAdd((x - px + n) % n);
}

struct SequenceBuilder {
    int len;
    int value[LIMIT];
    vector<int> ops;

    bool operator < (const SequenceBuilder &rhs) const {
        for (int i = 0; i < len; ++i) {
            if (value[i] != rhs.value[i]) {
                return value[i] < rhs.value[i];
            }
        }
        return false;
    }

    bool isNondecreasing() const {
        for (int i = 0; i + 1 < len; ++i) {
            if (value[i] > value[i + 1]) {
                return false;
            }
        }
        return true;
    }

    SequenceBuilder getInverse() const {
        SequenceBuilder ret;
        ret.len = len;

        for (int i = 0; i < len; ++i) {
            ret.value[value[i]] = i;
        }

        return ret;
    }

    bool buildOperations() {
        bool appeared[100005] = {};

        for (int i = 0; i < len; ++i) {
            appeared[i] = true;
        }

        for (int i = 0; i < len; ++i) {
            if (!appeared[i]) {
                return false;
            }
        }

        if (len == 1) {
            return true;
        }

        SequenceBuilder evenPart;
        SequenceBuilder oddPart;

        evenPart.len = len / 2;
        oddPart.len = len / 2;

        for (int i = 0; i < len / 2; ++i) {
            evenPart.value[i] = value[i * 2] / 2;
            oddPart.value[i] = value[i * 2 + 1] / 2;
        }

        if (!evenPart.buildOperations()) {
            return false;
        }

        if (!oddPart.buildOperations()) {
            return false;
        }

        if (value[0] & 1) {
            ops.push_back(len == 2 ? 1 : -1);
        }

        int leftMask = 0;

        for (int op : evenPart.ops) {
            if (op > 0) {
                ops.push_back(-1);
                ops.push_back(1);
            } else {
                ops.push_back(op * 2);
                leftMask ^= (-op) * 2;
            }
        }

        if (leftMask != 0) {
            ops.push_back(-leftMask);
        }

        int rightMask = 0;

        for (int op : oddPart.ops) {
            if (op > 0) {
                ops.push_back(1);
                ops.push_back(-1);
            } else {
                ops.push_back(op * 2);
                rightMask ^= (-op) * 2;
            }
        }

        if ((rightMask & (len / 2)) != (leftMask & (len / 2))) {
            return false;
        }

        if (leftMask >= len / 2) {
            leftMask -= len / 2;
        }

        if (rightMask >= len / 2) {
            rightMask -= len / 2;
        }

        if (leftMask != rightMask) {
            return false;
        }

        vector<int> compressed;

        for (int op : ops) {
            if (compressed.empty()) {
                compressed.push_back(op);
                continue;
            }

            if (op < 0 && compressed.back() < 0) {
                compressed.back() = -((-compressed.back()) ^ (-op));

                if (compressed.back() == 0) {
                    compressed.pop_back();
                }
            } else {
                compressed.push_back(op);
            }
        }

        ops.swap(compressed);
        return true;
    }
};

bool prepareByLowBits() {
    if (blockSize <= 1) {
        return true;
    }

    SequenceBuilder builder;
    builder.len = blockSize;

    for (int i = 0; i < n; ++i) {
        builder.value[i] = permArr[i] & (blockSize - 1);
    }

    if (!builder.buildOperations()) {
        return false;
    }

    for (int op : builder.ops) {
        if (op > 0) {
            doAdd(op);
        } else {
            doXor(-op);
        }
    }

    return true;
}

bool checkAndFixResidueClasses() {
    for (int r = 0; r < blockSize; ++r) {
        vector<int> group;

        for (int i = r; i < n; i += blockSize) {
            group.push_back(permArr[i]);
        }

        sort(group.begin(), group.end());

        int idx = 0;
        bool valid = true;

        for (int i = r; i < n; i += blockSize) {
            if (group[idx] != i) {
                valid = false;
                break;
            }
            ++idx;
        }

        if (!valid) {
            return false;
        }

        for (int i = r; i < n; i += blockSize) {
            if (permArr[i] != i) {
                swapByOperations(i, permArr[i]);
            }
        }
    }

    return true;
}

int main() {
    n = fastRead();
    a = fastRead();
    b = fastRead();

    for (int i = 0; i < n; ++i) {
        permArr[i] = fastRead();
    }

    refreshPosition();

    blockSize = (a - b + n) % n;
    blockSize &= -blockSize;

    if (blockSize == 0) {
        blockSize = n;
    }

    if (!prepareByLowBits()) {
        printf("-1\n");
        return 0;
    }

    if (!checkAndFixResidueClasses()) {
        printf("-1\n");
        return 0;
    }

    for (int i = 0; i < n; ++i) {
        assert(permArr[i] == i);
    }

    printf("%d\n", (int)resultOps.size());

    for (int op : resultOps) {
        if (op == 0) {
            printf("0\n");
        } else if (op < 0) {
            printf("1 %d\n", -op);
        } else {
            printf("2 %d\n", op);
        }
    }

    return 0;
}

SyntaxError: invalid syntax (1979102704.py, line 2)

## B 长跑

In [ ]:
	#include <iostream>
	#include <vector>
	#include <queue>
	#include <algorithm>
	using namespace std;
	struct Station {
	    int P; // 位置
	    int C; // 花费
	};
	int main() {
	    ios_base::sync_with_stdio(false);
	    cin.tie(NULL);
	    int N, L, Maxn, S;
	    while (cin >> N >> L >> Maxn >> S) {
	        vector<Station> stations(N);
	        for (int i = 0; i < N; ++i) {
	            cin >> stations[i].P >> stations[i].C;
	        }
	        sort(stations.begin(), stations.end(), [](const Station& a, const Station& b) {
	            return a.P < b.P;
	        });
	        if (Maxn >= L) {
	            cout << "Yes\n";
	            continue;
	        }
	        // visited[i][s] 记录在补给站i、剩余硬币为s的状态是否已访问
	        vector<vector<bool>> visited(N, vector<bool>(S + 1, false));
	        queue<pair<int, int>> q;
	        for (int i = 0; i < N; ++i) {
	            if (stations[i].P <= Maxn) {
	                if (S >= stations[i].C) {
	                    int rem = S - stations[i].C;
	                    if (!visited[i][rem]) {
	                        visited[i][rem] = true;
	                        q.push({i, rem});
	                    }
	                }
	            } else {
	                break; 
	            }
	        }
	        bool success = false;
	        while (!q.empty()) {
	            auto [u, s] = q.front();
	            q.pop();
	            int pos = stations[u].P;
	            if (pos + Maxn >= L) {
	                success = true;
	                break;
	            }
	            for (int v = u + 1; v < N; ++v) {
	                if (stations[v].P - pos <= Maxn) {
	                    if (s >= stations[v].C) {
	                        int rem = s - stations[v].C;
	                        if (!visited[v][rem]) {
	                            visited[v][rem] = true;
	                            q.push({v, rem});
	                        }
	                    }
	                } else {
	                    break;
	                }
	            }
	        }
	        cout << (success ? "Yes" : "No") << "\n";
	    }
	    return 0;
	}

## C 最长回文

In [ ]:
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>

using namespace std;

typedef unsigned long long ull;

const int MAXN = 100005;
const ull BASE = 13331;

ull p[MAXN];
ull ha[MAXN], rha[MAXN];
ull hb[MAXN], rhb[MAXN];

void init_hash(const string& A, const string& B, int n) {
    p[0] = 1;
    for (int i = 1; i <= n; ++i) {
        p[i] = p[i - 1] * BASE;
    }
    for (int i = 1; i <= n; ++i) {
        ha[i] = ha[i - 1] * BASE + A[i - 1];
        hb[i] = hb[i - 1] * BASE + B[i - 1];
    }
    for (int i = n; i >= 1; --i) {
        rha[i] = rha[i + 1] * BASE + A[i - 1];
        rhb[i] = rhb[i + 1] * BASE + B[i - 1];
    }
}

ull get_hash_a(int l, int r) {
    if (l > r) return 0;
    return ha[r] - ha[l - 1] * p[r - l + 1];
}

ull get_rhash_a(int l, int r) {
    if (l > r) return 0;
    return rha[l] - rha[r + 1] * p[r - l + 1];
}

ull get_hash_b(int l, int r) {
    if (l > r) return 0;
    return hb[r] - hb[l - 1] * p[r - l + 1];
}

ull get_rhash_b(int l, int r) {
    if (l > r) return 0;
    return rhb[l] - rhb[r + 1] * p[r - l + 1];
}

int LCP(int i, int j, int n) {
    if (i <= 0 || j > n) return 0;
    int limit = min(i, n - j + 1);
    int low = 1, high = limit, best = 0;
    while (low <= high) {
        int mid = low + (high - low) / 2;
        if (get_rhash_a(i - mid + 1, i) == get_hash_b(j, j + mid - 1)) {
            best = mid;
            low = mid + 1;
        } else {
            high = mid - 1;
        }
    }
    return best;
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    
    int n;
    if (!(cin >> n)) return 0;
    string A, B;
    cin >> A >> B;
    
    init_hash(A, B, n);
    int max_len = 0;

    for (int i = 1; i <= n; ++i) {
        int low = 0, high = min(i - 1, n - i), k = 0;
        while (low <= high) {
            int mid = low + (high - low) / 2;
            if (get_hash_a(i - mid, i - 1) == get_rhash_a(i + 1, i + mid)) {
                k = mid;
                low = mid + 1;
            } else {
                high = mid - 1;
            }
        }
        int R = i + k, L = i - k;
        int ext = LCP(L - 1, R, n);
        max_len = max(max_len, (R - L + 1) + 2 * ext);

        if (i < n && A[i - 1] == A[i]) {
            low = 1; high = min(i, n - i); k = 0;
            while (low <= high) {
                int mid = low + (high - low) / 2;
                if (get_hash_a(i - mid + 1, i) == get_rhash_a(i + 1, i + mid)) {
                    k = mid;
                    low = mid + 1;
                } else {
                    high = mid - 1;
                }
            }
            R = i + k; L = i - k + 1;
            ext = LCP(L - 1, R, n);
            max_len = max(max_len, (R - L + 1) + 2 * ext);
        }
    }

    for (int i = 1; i <= n; ++i) {
        int low = 0, high = min(i - 1, n - i), k = 0;
        while (low <= high) {
            int mid = low + (high - low) / 2;
            if (get_hash_b(i - mid, i - 1) == get_rhash_b(i + 1, i + mid)) {
                k = mid;
                low = mid + 1;
            } else {
                high = mid - 1;
            }
        }
        int R = i + k, L = i - k;
        int ext = LCP(L, R + 1, n);
        max_len = max(max_len, (R - L + 1) + 2 * ext);

        if (i < n && B[i - 1] == B[i]) {
            low = 1; high = min(i, n - i); k = 0;
            while (low <= high) {
                int mid = low + (high - low) / 2;
                if (get_hash_b(i - mid + 1, i) == get_rhash_b(i + 1, i + mid)) {
                    k = mid;
                    low = mid + 1;
                } else {
                    high = mid - 1;
                }
            }
            R = i + k; L = i - k + 1;
            ext = LCP(L, R + 1, n);
            max_len = max(max_len, (R - L + 1) + 2 * ext);
        }
    }

    for (int x = 1; x <= n; ++x) {
        int ext = LCP(x, x, n);
        max_len = max(max_len, 2 * ext);
    }

    cout << max_len << "\n";
    return 0;
}

## D 优惠券

In [ ]:
#include <iostream>
#include <vector>
#include <string>

using namespace std;

const int MAX_X = 100005;

struct BIT {
    int n;
    vector<int> tree;
    
    BIT(int n) : n(n), tree(n + 1, 0) {}

    void add(int i, int delta) {
        for (; i <= n; i += i & -i) {
            tree[i] += delta;
        }
    }

    int query(int i) {
        int sum = 0;
        for (; i > 0; i -= i & -i) {
            sum += tree[i];
        }
        return sum;
    }


    int find_kth(int k) {
        int ans = 0;
        for (int i = 19; i >= 0; i--) {
            int next_idx = ans + (1 << i);
            if (next_idx <= n && tree[next_idx] < k) {
                ans = next_idx;
                k -= tree[next_idx];
            }
        }
        return ans + 1;
    }
};


bool owned[MAX_X];
int last_pos[MAX_X];

void solve() {
    int m;
    while (cin >> m) {
        if (m == 0) {
            cout << -1 << "\n";
            continue;
        }

        BIT bit(m);
        vector<int> touched; 

        int ans = -1;
        bool failed = false;

        for (int pos = 1; pos <= m; ++pos) {
            string op;
            cin >> op;

        
            if (op == "I" || op == "O") {
                int x;
                cin >> x;
                
                
                if (failed) continue;

                touched.push_back(x);

                if (op == "I") {
                    if (owned[x]) {
                        
                        int L = last_pos[x];
                        int S = bit.query(L);
                        int idx = bit.find_kth(S + 1);
                        
                        if (idx < pos) {
                            bit.add(idx, -1); 
                            last_pos[x] = pos;
                        } else {
                            ans = pos;
                            failed = true;
                        }
                    } else {
                        owned[x] = true;
                        last_pos[x] = pos;
                    }
                } else if (op == "O") {
                    if (!owned[x]) {
                        int L = last_pos[x];
                        int S = bit.query(L);
                        int idx = bit.find_kth(S + 1);
                        
                        if (idx < pos) {
                            bit.add(idx, -1); 
                            last_pos[x] = pos;
                        } else {
                            ans = pos;
                            failed = true;
                        }
                    } else {
                        owned[x] = false;
                        last_pos[x] = pos;
                    }
                }
            } else {
                if (!failed) {
                    bit.add(pos, 1);
                }
            }
        }

        cout << ans << "\n";

        for (int x : touched) {
            owned[x] = false;
            last_pos[x] = 0;
        }
    }
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    
    solve();
    
    return 0;
}

## E 任意点

In [ ]:
#include <iostream>
#include <vector>

using namespace std;

struct Point {
    int x, y;
};

struct DSU {
    vector<int> parent;
    int count; // 记录连通块的数量

    DSU(int n) {
        parent.resize(n);
        for (int i = 0; i < n; i++) {
            parent[i] = i;
        }
        count = n; 
    }

    int find(int i) {
        if (parent[i] == i)
            return i;
        return parent[i] = find(parent[i]); 
    }

    void unite(int i, int j) {
        int root_i = find(i);
        int root_j = find(j);
        if (root_i != root_j) {
            parent[root_i] = root_j;
            count--; 
        }
    }
};

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;

    vector<Point> points(n);
    for (int i = 0; i < n; i++) {
        cin >> points[i].x >> points[i].y;
    }

    DSU dsu(n);

    for (int i = 0; i < n; i++) {
        for (int j = i + 1; j < n; j++) {
            if (points[i].x == points[j].x || points[i].y == points[j].y) {
                dsu.unite(i, j);
            }
        }
    }

    cout << dsu.count - 1 << "\n";

    return 0;
}

## F 通配符匹配

In [ ]:
#include <iostream>
#include <vector>
#include <string>

using namespace std;


enum OpType {
    STAR,    // '*'
    QMARK,   // '?'
    LITERAL  // '纯字串'
};

struct Operation {
    OpType type;
    string s;
    vector<int> pi; 
};


vector<int> build_kmp(const string& p) {
    int m = p.length();
    vector<int> pi(m, 0);
    for (int i = 1, j = 0; i < m; i++) {
        while (j > 0 && p[i] != p[j]) j = pi[j - 1];
        if (p[i] == p[j]) j++;
        pi[i] = j;
    }
    return pi;
}


vector<bool> find_matches(const string& t, const string& p, const vector<int>& pi) {
    int n = t.length();
    int m = p.length();
    vector<bool> match(n, false);
    if (m == 0) return match;
    
    for (int i = 0, j = 0; i < n; i++) {
        while (j > 0 && t[i] != p[j]) j = pi[j - 1];
        if (t[i] == p[j]) j++;
        if (j == m) {
            match[i - m + 1] = true;
            j = pi[j - 1];
        }
    }
    return match;
}

int main() {
   
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    string pattern;
    if (!(cin >> pattern)) return 0;

    int n;
    cin >> n;

    
    vector<Operation> ops;
    string cur_literal = "";

    for (char c : pattern) {
        if (c == '*' || c == '?') {
            
            if (!cur_literal.empty()) {
                Operation op;
                op.type = LITERAL;
                op.s = cur_literal;
                op.pi = build_kmp(cur_literal);
                ops.push_back(op);
                cur_literal = "";
            }
            
            if (c == '*') {
                
                if (ops.empty() || ops.back().type != STAR) {
                    Operation op;
                    op.type = STAR;
                    ops.push_back(op);
                }
            } else { // c == '?'
                Operation op;
                op.type = QMARK;
                ops.push_back(op);
            }
        } else {
            cur_literal += c;
        }
    }
    
    if (!cur_literal.empty()) {
        Operation op;
        op.type = LITERAL;
        op.s = cur_literal;
        op.pi = build_kmp(cur_literal);
        ops.push_back(op);
    }

 
    for (int idx = 0; idx < n; idx++) {
        string t;
        cin >> t;
        int t_len = t.length();

       
        vector<bool> valid(t_len + 1, false);
        valid[0] = true;

        for (const Operation& op : ops) {
            vector<bool> next_valid(t_len + 1, false);
            
            if (op.type == STAR) {
                bool met = false;
                for (int i = 0; i <= t_len; i++) {
                    if (valid[i]) met = true;
                    if (met) next_valid[i] = true;
                }
            } else if (op.type == QMARK) {
               
                for (int i = 0; i < t_len; i++) {
                    if (valid[i]) {
                        next_valid[i + 1] = true;
                    }
                }
            } else if (op.type == LITERAL) {
                
                vector<bool> match = find_matches(t, op.s, op.pi);
                int m_length = op.s.length();
                
                for (int i = 0; i + m_length <= t_len; i++) {
                    if (valid[i] && match[i]) {
                        next_valid[i + m_length] = true;
                    }
                }
            }
            valid = next_valid;
        }

        if (valid[t_len]) {
            cout << "YES\n";
        } else {
            cout << "NO\n";
        }
    }

    return 0;
}

## G 汉诺塔

In [ ]:
#include <iostream>
#include <string>
#include <vector>
#include <unordered_map>

using namespace std;

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;

    vector<string> priority(6);
    for (int i = 0; i < 6; i++) {
        cin >> priority[i];
    }

    // dp 数组：
    // steps[i][j] 表示将堆叠在柱子 j 上的 i 个盘子整体全部移动到另一个柱子所需要的步数
    // dest[i][j] 表示将堆叠在柱子 j 上的 i 个盘子整体移动后，最终会停留在哪个柱子上
    long long steps[35][3];
    int dest[35][3];

    // 柱子映射 A -> 0, B -> 1, C -> 2
    for (int from = 0; from < 3; from++) {
        for (int p = 0; p < 6; p++) {
            int u = priority[p][0] - 'A';
            int v = priority[p][1] - 'A';
            if (u == from) {
                dest[1][from] = v;
                steps[1][from] = 1;
                break;
            }
        }
    }


    for (int i = 2; i <= n; i++) {
        for (int X = 0; X < 3; X++) {
            int Z = dest[i - 1][X];
            int Y = 3 - X - Z; 
            long long cur_steps = steps[i - 1][X] + 1; 
            
            int cur_Z = Z; 
            int cur_Y = Y; 

            while (true) {
                int next_Z = dest[i - 1][cur_Z]; 
                
                if (next_Z == cur_Y) {
                    // 如果小盘子追到了大盘子所在的柱子，说明合并完毕！
                    cur_steps += steps[i - 1][cur_Z];
                    dest[i][X] = cur_Y;
                    steps[i][X] = cur_steps;
                    break;
                } else {
                    // 如果小盘子到了另一半，那么大盘子被迫移走，然后小盘子继续追
                    cur_steps += steps[i - 1][cur_Z]; // 小盘子移动过去
                    int next_Y = 3 - cur_Y - next_Z; // 大盘子接着必须移动到非小盘子停留点
                    cur_steps += 1;                  // 大盘子移动的步数
                    
                    cur_Y = next_Y;
                    cur_Z = next_Z;
                }
            }
        }
    }

    cout << steps[n][0] << "\n";

    return 0;
}

## H 马步距离

In [ ]:
#include <iostream>
#include <cmath>
#include <algorithm>

using namespace std;

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    
    int xp, yp, xs, ys;
    if (!(cin >> xp >> yp >> xs >> ys)) return 0;
    
    int dx = abs(xs - xp);
    int dy = abs(ys - yp);
    
    if (dx < dy) {
        swap(dx, dy);
    }
    

    if (dx == 1 && dy == 0) {
        cout << 3 << "\n";
        return 0;
    }
    if (dx == 2 && dy == 2) {
        cout << 4 << "\n";
        return 0;
    }
       
    int ans = max((dx + 1) / 2, (dx + dy + 2) / 3);
    if (ans % 2 != (dx + dy) % 2) {
        ans++;
    }
    
    cout << ans << "\n";
    
    return 0;
}

## I 直方图最大矩形

In [ ]:
class Solution {
public:
    /**
     * 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
     *
     * 
     * @param heights int整型vector 
     * @return int整型
     */
    int largestRectangleArea(vector<int>& heights) {
        // write code here
        stack<int> st;
        int max_area = 0;
        int n = heights.size();
        
        // 遍历到 n，当 i == n 时，利用高度 0 强制将栈内剩余的柱子全部计算并弹出
        for (int i = 0; i <= n; ++i) {
            int current_h = (i == n) ? 0 : heights[i];
            
            // 如果遇到比当前栈顶矮的柱子，说明栈顶柱子的右边界确定了
            while (!st.empty() && current_h < heights[st.top()]) {
                int h = heights[st.top()];
                st.pop();
                
                // 右边界是 i，左边界是弹出后的新栈顶
                // 如果栈为空，说明该柱子向左可以延伸到最左端（宽度就是 i）
                int w = st.empty() ? i : (i - st.top() - 1);
                
                max_area = max(max_area, h * w);
            }
            st.push(i);
        }
        
        return max_area;
    }
};

## J 消防局的设立

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

int main() {

    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;

    vector<int> p(n + 1, 0); // 父节点数组
    for (int i = 2; i <= n; i++) {
        cin >> p[i];
    }

    const int INF = 1e9;
    vector<int> req(n + 1, 0);   // 最远未覆盖节点距离，初始为 0
    vector<int> cov(n + 1, INF); // 最近消防局距离，初始为 INF
    int ans = 0;

    // 自底向上，根据拓扑序递减遍历
    for (int i = n; i >= 1; i--) {
        // 如果内部的最近消防局已经能覆盖最远未覆盖的需求，取消需求
        if (req[i] + cov[i] <= 2) {
            req[i] = -INF;
        }

        if (req[i] == 2) {
            ans++;
            cov[i] = 0;       // 当地建站，最近距离刷新为 0
            req[i] = -INF;  
        }

        // 把当前节点的情况向上传递给父节点
        int parent = p[i];
        if (parent >= 1) {
            // 父节点被需求的最大距离，至少是当前节点需求距离 +1
            req[parent] = max(req[parent], req[i] + 1);
            // 父节点看到的最近消防局，至多是当前节点最近消防局距离 +1
            cov[parent] = min(cov[parent], cov[i] + 1);
        }
    }

    if (req[1] >= 0) {
        ans++;
    }

    cout << ans << "\n";

    return 0;
}